In [10]:
from pydantic import BaseModel, Field
from typing import List
from enum import Enum

class GameGenre(str, Enum):
    ACTION = "Action"
    ADVENTURE = "Adventure"
    RPG = "RPG"
    RTS = "RTS"
    SHOOTER = "Shooter"
    SPORTS = "Sports"
    SURVIVAL = "Survival"

class Game(BaseModel):
    rank: int = Field(description="Rank of the game in the all-time best ranking; also used as game id")
    title: str = Field(description="Game title")
    genre: GameGenre = Field(description="Genre of the game")
    platforms: List[str] = Field(description="Platform(s) the game is available on, e.g. 'PC', 'Console', 'Mobile', etc.")
    release_date: str = Field(description="Release date of the game in YYYY-MM-DD format")
    developer: str = Field(description="Developer of the game")
    publisher: str = Field(description="Publisher of the game")
    description: str = Field(description="Brief description of the game")


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd


load_dotenv()
openai_client = OpenAI()
top_games = pd.read_csv("./top_games.csv")

PROMPT_TEMPLATE = """
Generate detailed information for the given video game.

Use the game id as a rank. Provide accurate information based on real game data.
Game decription should be no longer than 300 words.

Game id: {game_id}
Game title: '{game_title}'
""".strip()

def process_game(game: pd.Series) -> dict:
    game_id = game["id"]
    game_title = game["title"]
    print(f"Processing game: {game_id}. {game_title}")

    prompt = PROMPT_TEMPLATE.format(game_id=game_id, game_title=game_title)
    response = openai_client.responses.parse(
        model="gpt-5.4-mini",
        input=[{"role": "user", "content": prompt}],
        text_format=Game,
    )
    return response.output_parsed.model_dump()

with ThreadPoolExecutor(max_workers=5) as executor:
    results = [executor.submit(process_game, row) for _, row in top_games.iloc[:5].iterrows()]

games_data = [game.result() for game in results]
games_df = pd.DataFrame(games_data).sort_values("rank")

Processing game: 1. Clair Obscur: Expedition 33Processing game: 2. Silent Hill 2 (2001)

Processing game: 3. The Witcher 3: Wild Hunt - Blood and Wine
Processing game: 4. Metal Gear Solid 3: Snake Eater
Processing game: 5. Splatoon Raiders


In [61]:
games_df.to_csv("./games_data.csv", index=False)